# Dede Korkut sesi — kendi sesinden Piper ince ayarı
Hücreleri **sırayla** çalıştır (▶). Takıldığın yerin çıktısını Claude'a yapıştır.

Önce: *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seç.

In [ ]:
!nvidia-smi | head -15

## 1) Gerekli araçlar ve Drive

In [ ]:
!pip -q install faster-whisper
!apt -qq install -y ffmpeg > /dev/null
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Hakan_ses.mp4 dosyasını Drive'ın ana klasörüne (MyDrive) yükle; adı farklıysa burayı değiştir
VIDEO = '/content/drive/MyDrive/Hakan_ses.mp4'
import os; print(os.path.exists(VIDEO), os.path.getsize(VIDEO)//1_000_000 if os.path.exists(VIDEO) else 0, 'MB')

## 2) Konuşmayı yazıya dök ve cümlelere böl (≈15-25 dk)

In [ ]:
!git clone -q https://github.com/hakanuste/UMAY.git
!python3 UMAY/arac/ses_egit/hazirla.py "$VIDEO" /content/veri

In [ ]:
!head -30 /content/veri/metadata.csv
!wc -l /content/veri/metadata.csv

## 3) Yazıyı gözden geçir
Sol taraftaki **Dosyalar** simgesinden `veri/metadata.csv` dosyasına çift tıkla; yanlış yazılmış sözcükleri düzelt, bozuk satırları sil, kaydet. Bitince sonraki hücreyi çalıştır (Drive'a yedeklenir).

In [ ]:
!mkdir -p /content/drive/MyDrive/umay_ses
!cp -r /content/veri /content/drive/MyDrive/umay_ses/
print('yedeklendi: MyDrive/umay_ses/veri')

## 4) Piper eğitim aracı ve Türkçe başlangıç noktası (dfki)

In [ ]:
!git clone -q https://github.com/OHF-Voice/piper1-gpl.git
%cd /content/piper1-gpl
!pip -q install -e .[train]
!ls

In [ ]:
# Bu adım derleme isterse depodaki TRAINING.md'ye bak; çıktıyı Claude'a yapıştır
!bash build_monotonic_align.sh 2>&1 | tail -5

In [ ]:
!pip -q install huggingface_hub
!huggingface-cli download rhasspy/piper-checkpoints --repo-type dataset --include "tr/tr_TR/dfki/medium/*.ckpt" --local-dir /content/ckpt
!ls -la /content/ckpt/tr/tr_TR/dfki/medium/

## 5) İnce ayar (T4'te saatler sürer; kesilirse kaldığı yerden sürdürülür)

In [ ]:
import glob
CKPT = glob.glob('/content/ckpt/tr/tr_TR/dfki/medium/*.ckpt')[0]
!python3 -m piper.train fit \
  --data.voice_name hakan \
  --data.csv_path /content/veri/metadata.csv \
  --data.audio_dir /content/veri/wavs \
  --model.sample_rate 22050 \
  --data.espeak_voice tr \
  --data.cache_dir /content/cache \
  --data.config_path /content/cikti/config.json \
  --data.batch_size 16 \
  --ckpt_path "$CKPT"

## 6) ONNX'e çevir ve Drive'a koy

In [ ]:
import glob
son = sorted(glob.glob('/content/**/*.ckpt', recursive=True), key=os.path.getmtime)[-1]
print(son)
!python3 -m piper.train.export_onnx --checkpoint "$son" --output-file /content/hakan.onnx
!cp /content/hakan.onnx /content/drive/MyDrive/umay_ses/hakan.onnx
!cp /content/cikti/config.json /content/drive/MyDrive/umay_ses/hakan.onnx.json

Mac'te: `hakan.onnx` ve `hakan.onnx.json` dosyalarını `~/.local/share/umay/ses/` içine koy; uygulamada Ayarlar → Ses modeli.